<a href="https://colab.research.google.com/github/felipetavaressilvaoliveira-netizen/senacai/blob/main/analise_de_som_desafio.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [11]:
import kagglehub
import os

# 1. DOWNLOAD
path = kagglehub.dataset_download("soumendraprasad/musical-instruments-sound-dataset")
# The actual audio files for training are in a subfolder structure, typically:
# <download_path>/Train_submission/Train_submission/
# The metadata (labels) are in:
# <download_path>/Train_submission/Metadata_Train.csv

# Print path information for debugging
print("Dataset downloaded to:", path)
print("Content of download directory:", os.listdir(path))
print("Content of Train_submission directory:", os.listdir(os.path.join(path, "Train_submission")))


Using Colab cache for faster access to the 'musical-instruments-sound-dataset' dataset.
Dataset downloaded to: /kaggle/input/musical-instruments-sound-dataset
Content of download directory: ['Test_submission', 'Train_submission', 'Metadata_Train.csv', 'Metadata_Test.csv']
Content of Train_submission directory: ['Train_submission']


In [17]:
import numpy as np
import librosa
import tensorflow as tf
from sklearn.model_selection import train_test_split
import os
import pandas as pd # Import pandas for reading metadata

# Configurações de Memória
IMG_SIZE = (224, 224)
CLASSES = ['Sound_Guitar', 'Sound_Drum', 'Sound_Violin', 'Sound_Piano'] # Corrected class labels to match metadata
MAX_FILES_PER_CLASS = 60  # Reduzimos aqui para garantir que a RAM aguente

def audio_to_spectrogram(audio_path):
    # Carrega apenas 2 segundos para economizar muita RAM
    y, sr = librosa.load(audio_path, duration=2)
    S = librosa.feature.melspectrogram(y=y, sr=sr, n_mels=128)
    S_dB = librosa.power_to_db(S, ref=np.max)

    # Redimensiona e converte para RGB (3 canais) para o MobileNet
    img = tf.image.resize(np.expand_dims(S_dB, -1), IMG_SIZE)
    img = tf.image.grayscale_to_rgb(img)
    return (img - np.min(img)) / (np.max(img) - np.min(img))

# Define the base paths for audio files and metadata
# 'path' variable is expected to be available from the previous cell's execution.
# Based on typical Kaggle dataset structure and previous outputs:
audio_files_base_path = os.path.join(path, "Train_submission", "Train_submission")
metadata_file_path = os.path.join(path, "Metadata_Train.csv") # Corrected path for metadata file

x_data, y_data = [], []
label_to_int = {label: i for i, label in enumerate(CLASSES)}

# Load metadata to get file names and their corresponding labels
if not os.path.exists(metadata_file_path):
    print(f"Error: Metadata file not found at '{metadata_file_path}'. Cannot load data.")
else:
    metadata_df = pd.read_csv(metadata_file_path)
    print(f"Metadata loaded from: {metadata_file_path}")

    # Iterate through each class to load a limited number of files
    for label_str in CLASSES:
        class_df = metadata_df[metadata_df['Class'] == label_str].head(MAX_FILES_PER_CLASS)
        print(f"Lendo {len(class_df)} arquivos para a classe '{label_str}'...")

        for index, row in class_df.iterrows():
            file_name = row['FileName']
            full_audio_path = os.path.join(audio_files_base_path, file_name)

            if not os.path.exists(full_audio_path):
                print(f"Warning: Audio file not found at '{full_audio_path}'. Skipping.")
                continue

            try:
                spec = audio_to_spectrogram(full_audio_path)
                x_data.append(spec.numpy()) # Converte para numpy para economizar memória de GPU
                y_data.append(label_to_int[label_str])
            except Exception as e:
                print(f"Error processing {full_audio_path}: {e}. Skipping.")

X = np.array(x_data)
y = np.array(y_data)

if len(X) == 0:
    print("Error: No audio data was successfully loaded. Please check dataset paths and contents.")
    # Prevent ValueError from train_test_split with empty data
    X_train, X_test, y_train, y_test = np.array([]), np.array([]), np.array([]), np.array([])
else:
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y) # Added stratify

print(f"\nSucesso! {len(X)} imagens processadas.")
print(f"Formato do array X_train: {X_train.shape}")
print(f"Formato do array y_train: {y_train.shape}")
print(f"Formato do array X_test: {X_test.shape}")
print(f"Formato do array y_test: {y_test.shape}")


Metadata loaded from: /kaggle/input/musical-instruments-sound-dataset/Metadata_Train.csv
Lendo 60 arquivos para a classe 'Sound_Guitar'...
Lendo 60 arquivos para a classe 'Sound_Drum'...
Lendo 60 arquivos para a classe 'Sound_Violin'...
Lendo 60 arquivos para a classe 'Sound_Piano'...

Sucesso! 240 imagens processadas.
Formato do array X_train: (192, 224, 224, 3)
Formato do array y_train: (192,)
Formato do array X_test: (48, 224, 224, 3)
Formato do array y_test: (48,)


In [18]:
from tensorflow.keras import layers, models, applications

# Criando o modelo baseado em MobileNetV2 - nesse momento estamos usando o conceito MobileNetV2
base_model = applications.MobileNetV2(input_shape=(224, 224, 3), include_top=False, weights='imagenet')
base_model.trainable = False

model = models.Sequential([
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.3),
    layers.Dense(len(CLASSES), activation='softmax')
])

model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

print("Iniciando treinamento final...")
model.fit(X_train, y_train, epochs=10, validation_data=(X_test, y_test), batch_size=16)

Iniciando treinamento final...
Epoch 1/10
12/12 ━━━━━━━━━━━━━━━━━━━━ 22s 1s/step - accuracy: 0.2969 - loss: nan - val_accuracy: 0.2500 - val_loss: nan
Epoch 2/10
12/12 ━━━━━━━━━━━━━━━━━━━━ 10s 862ms/step - accuracy: 0.2500 - loss: nan - val_accuracy: 0.2500 - val_loss: nan
Epoch 3/10
12/12 ━━━━━━━━━━━━━━━━━━━━ 9s 736ms/step - accuracy: 0.2500 - loss: nan - val_accuracy: 0.2500 - val_loss: nan
Epoch 4/10
12/12 ━━━━━━━━━━━━━━━━━━━━ 10s 712ms/step - accuracy: 0.2500 - loss: nan - val_accuracy: 0.2500 - val_loss: nan
Epoch 5/10
12/12 ━━━━━━━━━━━━━━━━━━━━ 11s 752ms/step - accuracy: 0.2500 - loss: nan - val_accuracy: 0.2500 - val_loss: nan
Epoch 6/10
12/12 ━━━━━━━━━━━━━━━━━━━━ 9s 743ms/step - accuracy: 0.2500 - loss: nan - val_accuracy: 0.2500 - val_loss: nan
Epoch 7/10
12/12 ━━━━━━━━━━━━━━━━━━━━ 9s 731ms/step - accuracy: 0.2500 - loss: nan - val_accuracy: 0.2500 - val_loss: nan
Epoch 8/10
12/12 ━━━━━━━━━━━━━━━━━━━━ 8s 639ms/step - accuracy: 0.2500 - loss: nan - val_accuracy: 0.2500 - val_lo